# Self-RAG: A Dynamic Approach to Retrieval-Augmented Generation

## Overview

Self-RAG adds **self-evaluation** to the RAG pipeline. Instead of blindly retrieving and generating, it asks itself a series of questions at each step:

| Step | Question the system asks itself |
|---|---|
| 1. **Retrieval Decision** | "Do I even *need* to retrieve documents for this query?" |
| 2. **Document Retrieval** | Fetch top-k similar documents from the vector store |
| 3. **Relevance Check** | "Is each retrieved document actually *relevant* to the query?" |
| 4. **Response Generation** | Generate a response for each relevant context |
| 5. **Support Assessment** | "Is my response *supported* by the context, or did I hallucinate?" |
| 6. **Utility Evaluation** | "How *useful* is this response to the user?" (1–5 score) |
| 7. **Best Response Selection** | Pick the response with the best support + utility |

## Models Used

- **LLM**: `gemma3:12b` via Ollama (local)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)

<div style="text-align: center;">

<img src="./images/self_rag.svg" alt="Self RAG" style="width:80%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

---
## Step 1: Set Up the LLM and Embedding Model

In [2]:
llm = ChatOllama(model="gemma3:12b", max_tokens=1000, temperature=0)
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("LLM and embedding model ready")

LLM and embedding model ready


---
## Step 2: Load the PDF and Create a Vector Store

In [3]:
path = "data/Understanding_Climate_Change.pdf"

loader = PyPDFLoader(path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(splits, embedding_model)

print(f"Loaded {len(documents)} pages, split into {len(splits)} chunks")
print(f"Vector store created")

Loaded 33 pages, split into 97 chunks
Vector store created


---
## Step 3: Define the JSON Schemas and LLM Chains for Each Self-RAG Step

Self-RAG uses the LLM as a **judge** at multiple points. Each judgment has its own prompt and structured output schema.

We define 5 chains:
1. **Retrieval chain** — Should we retrieve? (Yes/No)
2. **Relevance chain** — Is this document relevant? (Relevant/Irrelevant)
3. **Generation chain** — Generate a response from context
4. **Support chain** — Is the response supported? (Fully/Partially/No support)
5. **Utility chain** — How useful is the response? (1–5)

In [4]:
# --- 1. Retrieval Decision ---
retrieval_schema = {
    "title": "RetrievalResponse",
    "type": "object",
    "properties": {
        "response": {"type": "string", "description": "Output only 'Yes' or 'No'"}
    },
    "required": ["response"]
}
retrieval_prompt = PromptTemplate(
    input_variables=["query"],
    template="Given the query '{query}', determine if retrieval is necessary. Output only 'Yes' or 'No'."
)
retrieval_chain = retrieval_prompt | llm.with_structured_output(retrieval_schema)

# --- 2. Relevance Evaluation ---
relevance_schema = {
    "title": "RelevanceResponse",
    "type": "object",
    "properties": {
        "response": {"type": "string", "description": "Output only 'Relevant' or 'Irrelevant'"}
    },
    "required": ["response"]
}
relevance_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Given the query '{query}' and the context '{context}', determine if the context is relevant. Output only 'Relevant' or 'Irrelevant'."
)
relevance_chain = relevance_prompt | llm.with_structured_output(relevance_schema)

# --- 3. Response Generation ---
generation_schema = {
    "title": "GenerationResponse",
    "type": "object",
    "properties": {
        "response": {"type": "string", "description": "The generated response"}
    },
    "required": ["response"]
}
generation_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Given the query '{query}' and the context '{context}', generate a response."
)
generation_chain = generation_prompt | llm.with_structured_output(generation_schema)

# --- 4. Support Assessment ---
support_schema = {
    "title": "SupportResponse",
    "type": "object",
    "properties": {
        "response": {"type": "string", "description": "Output 'Fully supported', 'Partially supported', or 'No support'"}
    },
    "required": ["response"]
}
support_prompt = PromptTemplate(
    input_variables=["response", "context"],
    template="Given the response '{response}' and the context '{context}', determine if the response is supported by the context. Output 'Fully supported', 'Partially supported', or 'No support'."
)
support_chain = support_prompt | llm.with_structured_output(support_schema)

# --- 5. Utility Evaluation ---
utility_schema = {
    "title": "UtilityResponse",
    "type": "object",
    "properties": {
        "response": {"type": "integer", "description": "Rate the utility of the response from 1 to 5"}
    },
    "required": ["response"]
}
utility_prompt = PromptTemplate(
    input_variables=["query", "response"],
    template="Given the query '{query}' and the response '{response}', rate the utility of the response from 1 to 5."
)
utility_chain = utility_prompt | llm.with_structured_output(utility_schema)

print("All 5 Self-RAG chains ready")

All 5 Self-RAG chains ready


---
---
# Test 1: High-Relevance Query

We test with a query that the document should clearly answer: *"What is the impact of climate change on the environment?"*

---
## Step 4: Define the Query

In [5]:
query = "What is the impact of climate change on the environment?"
print(f"Query: {query}")

Query: What is the impact of climate change on the environment?


---
## Step 5: Self-RAG — Is Retrieval Necessary?

The first self-check: does this query need retrieval, or can the LLM answer it directly?

In [6]:
retrieval_decision = retrieval_chain.invoke({"query": query})["response"].strip().lower()
print(f"Retrieval needed? {retrieval_decision}")

Retrieval needed? yes


---
## Step 6: Retrieve Documents

Since retrieval is needed, we fetch the top-3 most similar chunks.

In [7]:
top_k = 3
docs = vectorstore.similarity_search(query, k=top_k)
contexts = [doc.page_content for doc in docs]

print(f"Retrieved {len(contexts)} documents:")
for i, ctx in enumerate(contexts, 1):
    print(f"\n  Doc {i}: {ctx[:150]}...")

Retrieved 3 documents:

  Doc 1: Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Fore...

  Doc 2: development of eco-friendly fertilizers and farming techniques is essential for reducing the 
agricultural sector's carbon footprint. 
Chapter 3: Effe...

  Doc 3: cultural perceptions. 
Youth Engagement 
Youth are vital stakeholders in climate action. Empowering young people through education, 
activism, and lea...


---
## Step 7: Self-RAG — Check Relevance of Each Document

For each retrieved document, the LLM judges: *"Is this relevant to the query?"*

Only relevant documents proceed to the generation step.

In [8]:
relevant_contexts = []

for i, context in enumerate(contexts, 1):
    relevance = relevance_chain.invoke({"query": query, "context": context})["response"].strip().lower()
    print(f"Doc {i}: {relevance}")
    if relevance == "relevant":
        relevant_contexts.append(context)

print(f"\n{len(relevant_contexts)} out of {len(contexts)} documents are relevant")

Doc 1: relevant
Doc 2: relevant
Doc 3: relevant

3 out of 3 documents are relevant


---
## Step 8: Generate, Assess Support, and Evaluate Utility

For **each relevant context**, we:
1. **Generate** a response.
2. **Check support** — is the response grounded in the context? (Fully / Partially / No support)
3. **Rate utility** — how useful is this response? (1–5)

We collect all candidates, then pick the best one.

In [9]:
if not relevant_contexts:
    print("No relevant contexts found. Generating without retrieval...")
    final_response = generation_chain.invoke({"query": query, "context": "No relevant context found."})["response"]
    print(f"\nResponse: {final_response}")
else:
    responses = []

    for i, context in enumerate(relevant_contexts, 1):
        print(f"{'='*50}")
        print(f"Context {i}")
        print(f"{'='*50}")

        # Generate
        response = generation_chain.invoke({"query": query, "context": context})["response"]
        print(f"Response: {response[:200]}...")

        # Assess support
        support = support_chain.invoke({"response": response, "context": context})["response"].strip().lower()
        print(f"Support: {support}")

        # Evaluate utility
        utility = int(utility_chain.invoke({"query": query, "response": response})["response"])
        print(f"Utility: {utility}/5")

        responses.append((response, support, utility))

    print(f"\n{'='*50}")
    print(f"Collected {len(responses)} candidate responses")

Context 1
Response: Climate change is significantly impacting the environment, affecting both terrestrial and marine ecosystems. On land, we're seeing shifts in where plants and animals live, changing the makeup of fores...
Support: climate change is significantly impacting the environment, affecting both terrestrial and marine ecosystems. on land, we're seeing shifts in where plants and animals live, changing the makeup of forests, grasslands, and deserts. this leads to a loss of biodiversity and disrupts the delicate balance of these ecosystems. in the oceans, rising temperatures, increased acidity, and altered currents are harming marine life, from coral reefs to deep-sea habitats. these changes are impacting marine food webs, fisheries, and overall biodiversity. essentially, climate change is causing widespread changes in species distribution, reproductive cycles, and overall ecosystem health in both land and sea environments.
Utility: 5/5
Context 2
Response: According to the provi

---
## Step 9: Select the Best Response

We pick the response with the best combination of:
1. Support level ("fully supported" is best)
2. Utility score (highest wins)

In [10]:
if relevant_contexts:
    best_response = max(responses, key=lambda x: (x[1] == "fully supported", x[2]))

    print(f"Best response:")
    print(f"  Support: {best_response[1]}")
    print(f"  Utility: {best_response[2]}/5")
    print(f"\nAnswer: {best_response[0]}")
    final_response = best_response[0]
else:
    print(f"Answer: {final_response}")

Best response:
  Support: fully supported
  Utility: 4/5

Answer: According to the provided text, climate change is already impacting the environment and these effects are expected to worsen. Key impacts include: 

*   **Rising Temperatures:** Global temperatures have risen by approximately 1.2 degrees Celsius (2.2 degrees Fahrenheit) since the late 19th century, with some regions experiencing more significant increases.
*   **More Frequent and Severe Heatwaves:** Heatwaves are becoming more common and intense, posing risks to human health, agriculture, and infrastructure, particularly in urban areas.
*   **Changing Seasons:** Climate change is altering the timing and length of seasons, impacting ecosystems and human activities.

The text also highlights the importance of developing eco-friendly fertilizers and farming techniques to reduce the agricultural sector's carbon footprint, suggesting that agriculture is both affected by and contributes to climate change.


---
---
# Test 2: Irrelevant Query (Harry Potter)

Now let's test with a query that has **nothing to do** with the climate change document: *"How did Harry beat Quirrell?"*

Self-RAG should retrieve documents, find them all **irrelevant**, and generate without retrieval.

---
## Step 10: Run Self-RAG on an Irrelevant Query

In [11]:
query2 = "how did harry beat quirrell?"
print(f"Query: {query2}\n")

# Step 1: Retrieval decision
retrieval_decision2 = retrieval_chain.invoke({"query": query2})["response"].strip().lower()
print(f"Retrieval needed? {retrieval_decision2}")

if retrieval_decision2 == "yes":
    # Step 2: Retrieve
    docs2 = vectorstore.similarity_search(query2, k=3)
    contexts2 = [doc.page_content for doc in docs2]
    print(f"Retrieved {len(contexts2)} documents")

    # Step 3: Check relevance
    relevant_contexts2 = []
    for i, context in enumerate(contexts2, 1):
        relevance = relevance_chain.invoke({"query": query2, "context": context})["response"].strip().lower()
        print(f"  Doc {i}: {relevance}")
        if relevance == "relevant":
            relevant_contexts2.append(context)

    print(f"\nRelevant contexts: {len(relevant_contexts2)}")

    if not relevant_contexts2:
        print("\nNo relevant contexts — generating without retrieval...")
        final_response2 = generation_chain.invoke({"query": query2, "context": "No relevant context found."})["response"]
    else:
        # Would do the full generation + support + utility pipeline here
        final_response2 = "(relevant contexts found — would proceed with generation)"
else:
    print("No retrieval needed — generating directly...")
    final_response2 = generation_chain.invoke({"query": query2, "context": "No retrieval necessary."})["response"]

print(f"\nFinal answer: {final_response2}")

Query: how did harry beat quirrell?

Retrieval needed? yes
Retrieved 3 documents
  Doc 1: irrelevant
  Doc 2: irrelevant
  Doc 3: irrelevant

Relevant contexts: 0

No relevant contexts — generating without retrieval...

Final answer: Based on the information available, I can't answer that question. The query 'how did harry beat quirrell?' requires context from the Harry Potter series. Since no relevant context was provided, I don't have the information to explain how Harry defeated Quirrell.


---
## Summary

| Self-RAG Step | What it does | Why it matters |
|---|---|---|
| Retrieval Decision | Decides if documents are needed | Avoids unnecessary retrieval |
| Relevance Check | Filters irrelevant documents | Prevents noise in generation |
| Support Assessment | Checks if response is grounded | Catches hallucination |
| Utility Evaluation | Rates usefulness (1–5) | Picks the most helpful answer |

**Test 1** (climate query): All documents were relevant, responses were generated, best one was selected by support + utility.

**Test 2** (Harry Potter query): All documents were irrelevant, so Self-RAG correctly skipped them and generated without retrieval — gracefully admitting it doesn't have the information.